# Differential Equations — Session 24
## Section 5.2: Linear Models — Boundary-Value Problems

**Planned length:** 90 minutes  
**Notebook type:** Student interactive lecture

### Learning objectives

Students should be able to:

1. derive the small-deflection beam equation;
2. translate support types into boundary conditions;
3. solve a uniform-load beam problem;
4. interpret bending moment, shear, slope, and deflection;
5. define eigenvalues and eigenfunctions;
6. derive the Dirichlet spectrum for $y''+\lambda y=0$;
7. visualize eigenmodes and orthogonality;
8. derive Euler's critical buckling loads.

**Edition:** Local Instructor Interactive Edition

> **Student interactive edition — local Jupyter/Cursor workflow**
>
> 1. Run the **Local notebook setup** cell below first.
> 2. Read each explanation and derivation in order.
> 3. Run simulation and visualization cells as you reach them.
> 4. At each **Classroom Checkpoint**, stop and work out your answer before class discussion continues.
> 5. This student edition intentionally contains **no instructor answer-reveal cells and no instructor solution notes**.
>
> **Tip:** During class, use `Shift + Enter` to move through the notebook one cell at a time.

In [ ]:
# Local notebook setup — run this cell first.
import importlib.util
import platform
import sys

_REQUIRED = ["numpy", "matplotlib", "scipy", "sympy", "ipywidgets"]
_missing = [name for name in _REQUIRED if importlib.util.find_spec(name) is None]

print(f"Python {sys.version.split()[0]} on {platform.system()}")
if _missing:
    print("Missing packages:", ", ".join(_missing))
    print("From the project folder, run:")
    print("python -m pip install -r requirements.txt")
else:
    print("Student notebook environment is ready.")

### Core 90-minute path

| Time | Topic |
|---:|---|
| 0–18 min | Beam assumptions and fourth-order model |
| 18–45 min | Boundary conditions and uniform-load deflection |
| 45–68 min | Eigenvalues and eigenfunctions |
| 68–85 min | Column buckling and critical loads |
| 85–90 min | Exit check |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sympy as sp
from scipy.integrate import solve_ivp, quad
from scipy.optimize import brentq
from IPython.display import display, Markdown

try:
    from ipywidgets import interact, FloatSlider, IntSlider, Dropdown
    WIDGETS_AVAILABLE = True
except ImportError:
    WIDGETS_AVAILABLE = False

plt.rcParams['figure.figsize'] = (8, 5)
plt.rcParams['axes.grid'] = True
np.set_printoptions(precision=6, suppress=True)

def solve_second_order(accel, t_span, y0, points=1000, **kwargs):
    def rhs(t, z):
        x, v = z
        return [v, accel(t, x, v)]
    t_eval = np.linspace(t_span[0], t_span[1], points)
    return solve_ivp(rhs, t_span, y0, t_eval=t_eval, **kwargs)

print('Notebook ready.')
print('Interactive widgets available:', WIDGETS_AVAILABLE)

## Formal theory reference

### Model 5.2-A — Euler–Bernoulli beam equation

Under the assumptions of a slender homogeneous beam, small deflection, and constant flexural rigidity $EI$,

$$
EIy^{(4)}(x)=w(x),
$$

where $w(x)$ is transverse load per unit length.

### Definition 5.2-B — Beam quantities

With the sign convention used here,

$$
M=EIy'',
\qquad
V=M'=EIy^{(3)},
$$

where $M$ is bending moment and $V$ is shear force.

### Boundary-condition dictionary

At an endpoint:

- clamped: $y=0$, $y'=0$;
- free: $y''=0$, $y^{(3)}=0$;
- simply supported: $y=0$, $y''=0$.

### Definition 5.2-C — Eigenvalue and eigenfunction

For a homogeneous boundary-value problem containing a parameter $\lambda$, an **eigenvalue** is a value of $\lambda$ for which a nontrivial solution exists. The corresponding nonzero solutions are **eigenfunctions**.

### Theorem 5.2-D — Dirichlet eigenpairs

For

$$
y''+\lambda y=0,
\qquad
y(0)=0,
\qquad
y(L)=0,
$$

the eigenvalues and eigenfunctions are

$$
\lambda_n=\left(\frac{n\pi}{L}\right)^2,
\qquad
y_n(x)=\sin\left(\frac{n\pi x}{L}\right),
\qquad n=1,2,\ldots
$$

### Proposition 5.2-E — Orthogonality

For $m\ne n$,

$$
\int_0^L y_m(x)y_n(x)\,dx=0.
$$

### Theorem 5.2-F — Euler buckling loads

For a pin-ended column,

$$
EIy''+Py=0,
\qquad y(0)=y(L)=0,
$$

nontrivial deflection occurs at

$$
P_n=EI\left(\frac{n\pi}{L}\right)^2.
$$

The smallest value

$$
P_1=\frac{\pi^2EI}{L^2}
$$

is the Euler critical load.

### Classroom Checkpoint — Simply Supported Beam

Which boundary conditions describe a simply supported beam at an endpoint in the Euler–Bernoulli model?

> Pause here. Before continuing, try to explain their reasoning before continuing.

## 1. Beam model and assumptions

For small slopes,

$$
\kappa\approx y''.
$$

The moment–curvature law $M=EI\kappa$ gives $M\approx EIy''$. Since $M''=w$, the deflection satisfies

$$
EIy^{(4)}=w.
$$

The fourth order requires four boundary conditions.

In [ ]:
# Original beam and support sketch
fig, ax = plt.subplots(figsize=(10,3.5))
ax.axis('off')
x=np.linspace(0.1,0.9,300)
y=0.55-0.12*(1-((x-0.5)/0.4)**2)**2
ax.plot([0.1,0.9],[0.65,0.65],linestyle='--',label='undeformed axis')
ax.plot(x,y,linewidth=3,label='deflection curve')
ax.plot([0.08,0.08],[0.35,0.8],linewidth=6)
ax.plot([0.92,0.92],[0.35,0.8],linewidth=6)
for xi in np.linspace(0.15,0.85,9):
    ax.annotate('',xy=(xi,0.60),xytext=(xi,0.88),arrowprops={'arrowstyle':'->'})
ax.text(0.5,0.95,'distributed load $w(x)$',ha='center')
ax.legend(loc='lower center',ncol=2)
plt.show()

## 2. Uniform load on a clamped–clamped beam

Let $w(x)=w_0$ and impose

$$
y(0)=y'(0)=y(L)=y'(L)=0.
$$

The solution is

$$
y(x)=\frac{w_0}{24EI}x^2(L-x)^2.
$$

The maximum deflection occurs at the midpoint.

In [ ]:
x,L,w0,EI=sp.symbols('x L w0 EI',positive=True)
y=w0*x**2*(L-x)**2/(24*EI)
print('fourth derivative residual:')
display(sp.simplify(EI*sp.diff(y,x,4)-w0))
print('boundary data:')
display([sp.simplify(y.subs(x,0)),sp.simplify(sp.diff(y,x).subs(x,0)),sp.simplify(y.subs(x,L)),sp.simplify(sp.diff(y,x).subs(x,L))])

In [ ]:
def beam_explorer(L=4.0, w0=2.0, EI=40.0, support='clamped-clamped'):
    x=np.linspace(0,L,600)
    if support=='clamped-clamped':
        y=w0*x**2*(L-x)**2/(24*EI)
    elif support=='simply-supported':
        y=w0*x*(L**3-2*L*x**2+x**3)/(24*EI)
    else: # cantilever clamped at x=0, free at L
        y=w0*x**2*(6*L**2-4*L*x+x**2)/(24*EI)
    plt.plot(x,y)
    plt.axhline(0,linestyle='--')
    plt.xlabel('x')
    plt.ylabel('downward deflection')
    plt.title(support)
    plt.show()
    print('maximum displayed deflection =',np.max(y))

if WIDGETS_AVAILABLE:
    interact(
        beam_explorer,
        L=FloatSlider(min=1,max=10,step=0.5,value=4),
        w0=FloatSlider(min=0.5,max=10,step=0.5,value=2),
        EI=FloatSlider(min=5,max=200,step=5,value=40),
        support=Dropdown(options=['clamped-clamped','simply-supported','cantilever'],value='clamped-clamped')
    )
else:
    beam_explorer()

### Deflection, slope, moment, and shear

Differentiation reveals distinct mechanical quantities. Their extrema generally occur at different locations.

In [ ]:
L,w0,EI=4.0,2.0,40.0
x=np.linspace(0,L,600)
y=w0*x**2*(L-x)**2/(24*EI)
dx=x[1]-x[0]
slope=np.gradient(y,dx)
moment=EI*np.gradient(slope,dx)
shear=np.gradient(moment,dx)
fig,ax=plt.subplots()
ax.plot(x,y,label='deflection')
ax.plot(x,slope,label='slope')
ax.plot(x,moment/np.max(np.abs(moment)),label='normalized moment')
ax.plot(x,shear/np.max(np.abs(shear)),label='normalized shear')
ax.legend(); ax.set_xlabel('x'); ax.set_title('Related beam quantities'); plt.show()

## 3. Eigenvalues and eigenfunctions

The boundary conditions select discrete parameter values. For $\lambda>0$, let $\lambda=\alpha^2$:

$$
y=c_1\cos\alpha x+c_2\sin\alpha x.
$$

The condition $y(0)=0$ gives $c_1=0$. A nontrivial second coefficient requires

$$
\sin(\alpha L)=0,
$$

so $\alpha L=n\pi$.

In [ ]:
def eigenmode_explorer(L=1.0, n=1, amplitude=1.0):
    x=np.linspace(0,L,700)
    y=amplitude*np.sin(n*np.pi*x/L)
    plt.plot(x,y)
    plt.axhline(0,linestyle='--')
    plt.scatter([0,L],[0,0])
    plt.xlabel('x')
    plt.ylabel('$y_n(x)$')
    plt.title(fr'$n={n}$, $\lambda_n={(n*np.pi/L)**2:.3f}$')
    plt.show()
    print('interior zeros =',n-1)

if WIDGETS_AVAILABLE:
    interact(
        eigenmode_explorer,
        L=FloatSlider(min=0.5,max=5,step=0.25,value=1),
        n=IntSlider(min=1,max=10,step=1,value=1),
        amplitude=FloatSlider(min=-2,max=2,step=0.1,value=1)
    )
else:
    eigenmode_explorer()

### Orthogonality matrix

Numerically computing the inner products makes the orthogonality pattern visible.

In [ ]:
L=1.0
x=np.linspace(0,L,4001)
N=6
G=np.zeros((N,N))
for m in range(1,N+1):
    for n in range(1,N+1):
        G[m-1,n-1]=np.trapezoid(np.sin(m*np.pi*x/L)*np.sin(n*np.pi*x/L),x)
plt.imshow(G)
plt.colorbar(label='inner product')
plt.xticks(range(N),range(1,N+1)); plt.yticks(range(N),range(1,N+1))
plt.xlabel('n'); plt.ylabel('m'); plt.title('Orthogonality matrix')
plt.show()
print(np.round(G,6))

## 4. Buckling of a thin column

The same eigenvalue problem describes a pin-ended column. The load $P$ plays the role of the spectral parameter through

$$
\lambda=\frac{P}{EI}.
$$

Below the first critical load, the ideal linear boundary-value problem has only the straight solution.

In [ ]:
def buckling_explorer(EI=100.0, L=3.0, mode=1, amplitude=0.2):
    Pcrit=EI*(mode*np.pi/L)**2
    x=np.linspace(0,L,700)
    y=amplitude*np.sin(mode*np.pi*x/L)
    plt.plot(y,x)
    plt.axvline(0,linestyle='--')
    plt.xlabel('lateral deflection')
    plt.ylabel('height')
    plt.title(fr'Buckling mode {mode}, critical load {Pcrit:.2f}')
    plt.show()
    print('critical load =',Pcrit)

if WIDGETS_AVAILABLE:
    interact(
        buckling_explorer,
        EI=FloatSlider(min=10,max=500,step=10,value=100),
        L=FloatSlider(min=1,max=10,step=0.5,value=3),
        mode=IntSlider(min=1,max=6,step=1,value=1),
        amplitude=FloatSlider(min=0.05,max=1,step=0.05,value=0.2)
    )
else:
    buckling_explorer()

## Classroom Checkpoint — Exit Check

For a simply supported beam endpoint, which two boundary conditions apply?

> Pause here. Let students commit to an answer before running the next cell.